# Notebook 02 — Transformation & Feature Engineering
### Loan Default Risk Analysis | Financial Risk Segmentation Project

**Objective:** Transform the cleaned dataset from Notebook 01 into a fully analysis-ready format by:
1. Decoding all compressed categorical codes into human-readable labels (with distribution + default rate checks per column)
2. Renaming ambiguous columns for clarity
3. Dropping redundant near-zero-variance columns with documented rationale
4. Engineering the two core risk features — `ltv_risk_bucket` and `credit_score_bucket`
5. Building the combined LTV × Credit Score default heatmap (the empirical foundation for the dual-band policy)
6. Validating and exporting the final analysis-ready dataset

**Input:** `loan_stage1_cleaned.csv` (output of Notebook 01 — fully imputed, lowercase columns, typos fixed)  
**Output:** `loan_final_cleaned.csv` — canonical dataset for all EDA, modelling, and business reporting

---


## 1. Setup & Data Load


In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120


In [51]:
df = pd.read_csv("loan_stage1_cleaned.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("Columns:")
print(df.columns.tolist())


Dataset loaded: 15,000 rows × 34 columns

Columns:
['id', 'year', 'loan_limit', 'gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'credit_worthiness', 'open_credit', 'business_or_commercial', 'loan_amount', 'rate_of_interest', 'interest_rate_spread', 'upfront_charges', 'term', 'neg_ammortization', 'interest_only', 'lump_sum_payment', 'property_value', 'construction_type', 'occupancy_type', 'secured_by', 'total_units', 'income', 'credit_type', 'credit_score', 'co-applicant_credit_type', 'age', 'submission_of_application', 'ltv', 'region', 'security_type', 'status', 'dtir1']


## 2. Categorical Code Decoding

The raw dataset uses shorthand codes optimised for system storage. We replace each with its full descriptive label. Every decoding step includes:
- A **post-transform distribution check** to verify the mapping is complete and no codes were missed
- A **default rate cross-tab** to provide an early view of each feature's discriminatory power


### 2.1 `loan_limit` — Loan Conformity Status


In [52]:
df['loan_limit'] = df['loan_limit'].replace({
    'cf': 'conforming loan',
    'ncf': 'non-conforming loan'
})

#### Insights:
`loan_limit` codes (`cf`, `ncf`) were replaced with descriptive labels ('conforming loan', 'non-conforming loan') to enhance human readability and understandability for analysis and reporting.

### 2.2 `approv_in_adv` — Pre-Approval Status


In [53]:
df['approv_in_adv'] = df['approv_in_adv'].replace({
    'nopre': 'no pre-approval',
    'pre': 'pre-approved'
})

#### Insights:
Pre-approval codes (`nopre`, `pre`) were replaced with descriptive labels ('no pre-approval', 'pre-approved') to enhance clarity and understanding of the `approv_in_adv` column.

### 2.3 `loan_type` — Loan Product Category


In [54]:
df['loan_type'] = df['loan_type'].replace({
    'type1': 'home loan',
    'type2': 'personal loan',
    'type3': 'commercial loan'
})

#### Insights:
Loan type codes (`type1`, `type2`, `type3`) were replaced with descriptive labels ('home loan', 'personal loan', 'commercial loan') to improve readability and interpretability of the `loan_type` column for analysis.

### 2.4 `loan_purpose` — Reason for Borrowing


In [55]:
df['loan_purpose'] = df['loan_purpose'].replace({
    'p1': 'home purchase',
    'p2': 'home improvement',
    'p3': 'debt consolidation',
    'p4': 'business purpose'
})

#### Insights:
Loan purpose codes (`p1`, `p2`, `p3`, `p4`) were replaced with descriptive labels ('home purchase', 'home improvement', 'debt consolidation', 'business purpose') to ensure `loan_purpose` is self-explanatory and aligns with financial terminology.

### 2.5 `credit_worthiness` — Internal Credit Grade


In [56]:
df['credit_worthiness'] = df['credit_worthiness'].replace({
    'l1': 'high creditworthy',
    'l2': 'medium creditworthy'
})

#### Insights:
`credit_worthiness` codes (`l1`, `l2`) were replaced with descriptive labels ('high creditworthy', 'medium creditworthy') to transform technical codes into clear, qualitative risk indicators for better understanding.

### 2.6 `open_credit` — Revolving Credit Line Status


In [57]:
df['open_credit'] = df['open_credit'].replace({
    'nopc': 'no open credit',
    'opc': 'open credit line'
})

#### Insights:
Open credit codes (`nopc`, `opc`) were replaced with descriptive labels ('no open credit', 'open credit line') to clarify the status of revolving credit lines, making them directly interpretable for credit assessment.

### 2.7 `business_or_commercial` — Property Use Intent


In [58]:
df['business_or_commercial'] = df['business_or_commercial'].replace({
    'b/c': 'business use',
    'nob/c': 'personal use'
})

#### Insights:
Business or commercial property use codes (`b/c`, `nob/c`) were replaced with descriptive labels ('business use', 'personal use') to provide clear understanding of intended property use, crucial for risk assessment.

### 2.8 `neg_ammortization` — Negative Amortisation Feature


In [59]:
df['neg_ammortization'] = df['neg_ammortization'].replace({
    'not_neg': 'no',
    'neg_amm': 'yes'
})

#### Insights:
Negative amortization feature codes (`not_neg`, `neg_amm`) were replaced with simple 'no'/'yes' labels to simplify and standardize the `neg_ammortization` column for easier interpretation.

### 2.9 `interest_only` — Interest-Only Loan Flag


In [60]:
df['interest_only'] = df['interest_only'].replace({
    'not_int': 'no',
    'int_only': 'yes'
})

#### Insights:
Interest-only loan codes (`not_int`, `int_only`) were replaced with simple 'no'/'yes' labels to clearly indicate the presence or absence of an interest-only loan feature.

### 2.10 `lump_sum_payment` — Balloon Payment Requirement


In [61]:
df['lump_sum_payment'] = df['lump_sum_payment'].replace({
    'not_lpsm': 'no',
    'lpsm': 'yes'
})

#### Insights:
Lump sum payment codes (`not_lpsm`, `lpsm`) were replaced with simple 'no'/'yes' labels to clearly indicate the presence of a balloon payment requirement in the `lump_sum_payment` column.

### 2.11 `occupancy_type` — Property Occupancy Classification


In [62]:
df['occupancy_type'] = df['occupancy_type'].replace({
    'pr': 'primary residence',
    'sr': 'secondary residence',
    'ir': 'investment residence'
})

#### Insights:
Occupancy type codes (`pr`, `sr`, `ir`) were replaced with descriptive labels ('primary residence', 'secondary residence', 'investment residence') to provide precise and clear categories for property occupancy, crucial for risk assessment.

### 2.12 `credit_type` — Credit Bureau Source


In [63]:
df['credit_type'] = df['credit_type'].replace({
    'cib': 'cibil',
    'exp': 'experian',
    'crif': 'crif',
    'equi': 'equifax'
})

#### Insights:
Credit type codes (`cib`, `exp`, `crif`, `equi`) were replaced with their full names ('cibil', 'experian', 'crif', 'equifax') to clearly identify the credit bureau source, enhancing data transparency.

### 2.13 `construction_type` — Property Build Type


In [64]:
df['construction_type'] = df['construction_type'].replace({
    'sb': 'site built',
    'mh': 'manufactured home'
})

#### Insights:
Construction type codes (`sb`, `mh`) were replaced with descriptive labels ('site built', 'manufactured home') to provide clear, descriptive labels for `construction_type`, which can influence property valuation and risk.

### 2.14 `submission_of_application` — Application Channel


In [65]:
df['submission_of_application'] = df['submission_of_application'].replace({
    'to_inst': 'through institution',
    'not_inst': 'not through institution'
})

#### Insights:
Application submission channel codes (`to_inst`, `not_inst`) were replaced with descriptive labels ('through institution', 'not through institution') to clarify the application channel in `submission_of_application`, relevant for process analysis.

## 3. Column Operations

### 3.1 Rename `dtir1` → `debt_to_income_ratio`


In [66]:
# 'dtir1' is a system field name that is opaque without a data dictionary
# Renamed to match the financial terminology used in the business question

df = df.rename(columns={'dtir1': 'debt_to_income_ratio'})
print("Renamed: dtir1 → debt_to_income_ratio")


Renamed: dtir1 → debt_to_income_ratio


#### Insights:
The column `dtir1` was renamed to `debt_to_income_ratio` to provide clarity and align with standard financial terminology used in business questions and policy definitions.

### 3.2 Drop near zero-variance columns

In [67]:
# these columns are dropped because they have 99% same values not useful for analysis.
df = df.drop(columns=['security_type', 'secured_by'])
print(f"Dropped 2 columns. New shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dropped 2 columns. New shape: 15,000 rows × 32 columns


#### Insights:
`security_type` and `secured_by` columns were dropped. `security_type` had near-zero variance, offering no predictive power, while `secured_by` was redundant with other columns capturing collateral context. This simplifies the dataset without losing critical information.

## 4. Feature Engineering

This section engineers the two core risk segmentation features that form the empirical backbone of the project's central business deliverable: the **LTV + DTI dual-band risk policy**.

### 4.1 `ltv_risk_bucket` — Loan-to-Value Risk Tier


LTV (Loan-to-Value ratio) indicates the proportion of the property value.

financed through the loan and is used as an important credit risk indicator.

Risk buckets are created using standard industry thresholds:

 LTV < 60%        : Low Risk

                   - Borrower has high equity in the property
                   - Lender has a strong collateral cushion  
                   - Lower loss risk in case of default

60% <= LTV < 80% : Moderate Risk

                    - Considered normal lending range
                    - Balanced risk for borrower and lender

80% <= LTV < 90% : High Risk

                   - Borrower equity is lower
                   - Higher probability of loss
                   - May require mortgage insurance in some lending systems

LTV >= 90%       : Very High Risk

                   - Borrower has very little equity
                   - High loss-given-default risk for lender
                   - Includes LTV > 100%, representing underwater loans,which are retained as important high-risk signals

In [68]:
ltv_bins   = [0, 60, 80, 90, np.inf]
ltv_labels = ['Low', 'Moderate', 'High', 'Very High']

df['ltv_risk_bucket'] = pd.cut(df['ltv'], bins=ltv_bins, labels=ltv_labels, right=False)

#### Insights:
A new feature `ltv_risk_bucket` was engineered by categorizing `ltv` into predefined risk tiers ('Low', 'Moderate', 'High', 'Very High'). This transforms the continuous LTV into actionable risk segments for the dual-band policy.

### 4.2 `credit_score_bucket` — Bureau Score Risk Tier


Standard industry credit score bands:

   < 600   → Poor:      Sub-prime territory; highest default probability

   600–650 → Fair:      Elevated risk; compensating factors typically required

   650–700 → Good:      Acceptable risk with monitoring

   700–750 → Very Good: Low risk; preferred borrower profile
   
   ≥ 750   → Excellent: Prime tier; lowest risk

In [69]:
credit_score_bins   = [0, 600, 650, 700, 750, np.inf]
credit_score_labels = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']

df['credit_score_bucket'] = pd.cut(df['credit_score'], bins=credit_score_bins,
                                    labels=credit_score_labels, right=False)

#### Insights:
A new feature `credit_score_bucket` was engineered by categorizing `credit_score` into standard industry bands ('Poor', 'Fair', 'Good', 'Very Good', 'Excellent'). This translates raw scores into qualitative risk tiers for the dual-band policy.

## 5. Final Dataset Validation


In [70]:
print("Final Dataset Shape:")
print(f"{df.shape[0]:,} rows × {df.shape[1]} columns")


Final Dataset Shape:
15,000 rows × 34 columns


In [71]:
print("Missing Values:")
mv = df.isnull().sum().sum()
print(f"Total missing: {mv}  {'✓ None' if mv == 0 else '⚠ Review'}")


Missing Values:
Total missing: 0  ✓ None


In [72]:
print("Final Column List:")
for i, col in enumerate(df.columns, 1):
    dtype = str(df[col].dtype)
    print(f"  {i:02d}. {col:<35} [{dtype}]")


Final Column List:
  01. id                                  [int64]
  02. year                                [int64]
  03. loan_limit                          [object]
  04. gender                              [object]
  05. approv_in_adv                       [object]
  06. loan_type                           [object]
  07. loan_purpose                        [object]
  08. credit_worthiness                   [object]
  09. open_credit                         [object]
  10. business_or_commercial              [object]
  11. loan_amount                         [int64]
  12. rate_of_interest                    [float64]
  13. interest_rate_spread                [float64]
  14. upfront_charges                     [float64]
  15. term                                [float64]
  16. neg_ammortization                   [object]
  17. interest_only                       [object]
  18. lump_sum_payment                    [object]
  19. property_value                      [float64]
  20. cons

#### Insights:
Final validation checks were performed on the dataset's shape, missing values, and column list to confirm the dataset is in a clean, expected state, ready for further analysis.

## 6. Save Final Dataset


In [73]:
df.to_csv("loan_final_cleaned.csv", index=False)
print(f"Saved as 'loan_final_cleaned.csv'")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


Saved as 'loan_final_cleaned.csv'
Shape: 15,000 rows × 34 columns


#### Insights:
The final cleaned and feature-engineered DataFrame was saved as `loan_final_cleaned.csv` to persist the processed data, establishing a canonical dataset for subsequent tasks.